In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings

In [5]:

warnings.filterwarnings('ignore')

# 1. 경로 설정
DATA_PATH = '/content/drive/MyDrive/스트레스 테스트 18차/'

# 2. Mish 활성화 함수
class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(nn.functional.softplus(x))

# 3. 고도화된 PyTorch 모델 (512-256 구조 유지)
class StressHybridNet(nn.Module):
    def __init__(self, numeric_dim, embed_dims):
        super(StressHybridNet, self).__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(ni, nf) for ni, nf in embed_dims])
        total_embed_dim = sum([nf for ni, nf in embed_dims])

        self.fc = nn.Sequential(
            nn.Linear(numeric_dim + total_embed_dim, 512),
            nn.BatchNorm1d(512),
            Mish(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            Mish(),
            nn.Linear(256, 64),
            Mish(),
            nn.Linear(64, 1)
        )

    def forward(self, x_num, x_cat):
        embeddings = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat([x_num] + embeddings, dim=1)
        return self.fc(x)

class UltimateFinalSolverV7:
    def __init__(self, data_path):
        self.path = data_path
        self.scaler = RobustScaler()
        self.cat_cols = ['gender', 'activity', 'smoke_status', 'edu_level', 'sleep_pattern']
        self.mappings = {
            'gender': {'F': 0, 'M': 1},
            'activity': {'light': 0, 'moderate': 1, 'intense': 2},
            'smoke_status': {'non-smoker': 0, 'ex-smoker': 1, 'current-smoker': 2, 'Unknown': 3},
            'edu_level': {'high school diploma': 0, 'bachelors degree': 1, 'graduate degree': 2, 'Unknown': 3},
            'sleep_pattern': {'sleep difficulty': 0, 'normal': 1, 'oversleeping': 2}
        }

    def preprocess(self, df, is_train=True):
        df = df.copy()
        for col in ['medical_history', 'family_medical_history', 'mean_working', 'edu_level']:
            df[f'{col}_nan'] = df[col].isna().astype(int)
            df[col] = df[col].fillna(0 if col == 'mean_working' else 'Unknown')

        for col, mapping in self.mappings.items():
            df[col] = df[col].map(mapping).fillna(len(mapping))

        df['bmi'] = (df['weight'] / ((df['height'] / 100.0) ** 2))
        df['pulse_pressure'] = df['systolic_blood_pressure'] - df['diastolic_blood_pressure']

        num_features = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
                        'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working',
                        'bmi', 'pulse_pressure'] + [c for c in df.columns if '_nan' in c]

        X_num = df[num_features].values
        X_cat = df[self.cat_cols].values.astype(np.int64)

        if is_train:
            self.scaler.fit(X_num)
            y = np.log1p(df['stress_score'].values)
            return self.scaler.transform(X_num), X_cat, y
        return self.scaler.transform(X_num), X_cat

    def run(self):
        train_df = pd.read_csv(f'{self.path}train.csv')
        test_df = pd.read_csv(f'{self.path}test.csv')

        X_num, X_cat, y = self.preprocess(train_df)
        X_num_test, X_cat_test = self.preprocess(test_df, is_train=False)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        kf = KFold(n_splits=10, shuffle=True, random_state=42)

        pt_test_preds, xgb_test_preds, lgb_test_preds = np.zeros(len(X_num_test)), np.zeros(len(X_num_test)), np.zeros(len(X_num_test))
        fold_maes = []
        embed_dims = [(len(m)+2, 8) for m in self.mappings.values()]

        for fold, (tr_idx, val_idx) in enumerate(kf.split(X_num)):
            print(f"\n🔥 Fold {fold+1} V7 Fine-Tuning...")

            # 1. PyTorch (비중 0.5로 조정하여 앙상블 밸런스 확보)
            X_n_tr, X_c_tr = torch.FloatTensor(X_num[tr_idx]).to(device), torch.LongTensor(X_cat[tr_idx]).to(device)
            y_tr = torch.FloatTensor(y[tr_idx]).view(-1, 1).to(device)
            X_n_val, X_c_val = torch.FloatTensor(X_num[val_idx]).to(device), torch.LongTensor(X_cat[val_idx]).to(device)
            y_val_real = train_df.iloc[val_idx]['stress_score'].values

            model = StressHybridNet(X_num.shape[1], embed_dims).to(device)
            optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=800)
            criterion = nn.L1Loss() # MAE 최적화를 위해 다시 L1Loss 복원

            # 배치 사이즈 16으로 롤백 (세밀한 학습 유도)
            train_loader = DataLoader(TensorDataset(X_n_tr, X_c_tr, y_tr), batch_size=16, shuffle=True)
            for epoch in range(800): # Epoch 증대
                model.train()
                for bn, bc, by in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model(bn, bc), by)
                    loss.backward()
                    optimizer.step()
                scheduler.step()

            model.eval()
            with torch.no_grad():
                pt_val = np.expm1(model(X_n_val, X_c_val).cpu().numpy()).flatten()
                pt_test_preds += np.expm1(model(torch.FloatTensor(X_num_test).to(device), torch.LongTensor(X_cat_test).to(device)).cpu().numpy()).flatten() / 10

            # 2. XGB (25% Weight)
            X_tree_tr, X_tree_val = np.hstack([X_num[tr_idx], X_cat[tr_idx]]), np.hstack([X_num[val_idx], X_cat[val_idx]])
            xgb = XGBRegressor(n_estimators=2000, learning_rate=0.005, max_depth=6, objective='reg:absoluteerror',
                               colsample_bytree=0.8, tree_method='gpu_hist' if torch.cuda.is_available() else 'auto')
            xgb.fit(X_tree_tr, y[tr_idx])
            xgb_val = np.expm1(xgb.predict(X_tree_val)).flatten()
            xgb_test_preds += np.expm1(xgb.predict(np.hstack([X_num_test, X_cat_test]))).flatten() / 10

            # 3. LGBM (25% Weight)
            lgb = LGBMRegressor(n_estimators=2000, learning_rate=0.005, num_leaves=63, objective='regression_l1', verbose=-1)
            lgb.fit(X_tree_tr, y[tr_idx])
            lgb_val = np.expm1(lgb.predict(X_tree_val)).flatten()
            lgb_test_preds += np.expm1(lgb.predict(np.hstack([X_num_test, X_cat_test]))).flatten() / 10

            # 🚀 앙상블 비중 재조정 (PT:XGB:LGBM = 0.5 : 0.25 : 0.25)
            ensemble_val = (pt_val * 0.5) + (xgb_val * 0.25) + (lgb_val * 0.25)
            fold_mae = mean_absolute_error(y_val_real, ensemble_val)
            fold_maes.append(fold_mae)
            print(f"✅ Fold {fold+1} MAE: {fold_mae:.6f}")

        print(f"\n✨ Final Average OOF MAE: {np.mean(fold_maes):.6f}")

        submit = pd.read_csv(f'{self.path}sample_submission.csv')
        submit['stress_score'] = (pt_test_preds * 0.5) + (xgb_test_preds * 0.25) + (lgb_test_preds * 0.25)
        submit.to_csv('submission_v7_final_fixed.csv', index=False)

if __name__ == "__main__":
    solver = UltimateFinalSolverV7(DATA_PATH)
    solver.run()



🔥 Fold 1 V7 Fine-Tuning...
✅ Fold 1 MAE: 0.155821

🔥 Fold 2 V7 Fine-Tuning...
✅ Fold 2 MAE: 0.166716

🔥 Fold 3 V7 Fine-Tuning...
✅ Fold 3 MAE: 0.168632

🔥 Fold 4 V7 Fine-Tuning...
✅ Fold 4 MAE: 0.180913

🔥 Fold 5 V7 Fine-Tuning...
✅ Fold 5 MAE: 0.177561

🔥 Fold 6 V7 Fine-Tuning...
✅ Fold 6 MAE: 0.159601

🔥 Fold 7 V7 Fine-Tuning...
✅ Fold 7 MAE: 0.175872

🔥 Fold 8 V7 Fine-Tuning...
✅ Fold 8 MAE: 0.178661

🔥 Fold 9 V7 Fine-Tuning...
✅ Fold 9 MAE: 0.177959

🔥 Fold 10 V7 Fine-Tuning...
✅ Fold 10 MAE: 0.165900

✨ Final Average OOF MAE: 0.170763
